# Autoresearch: Qwen 3.5 27B Optimization

Track how the model config evolves through autonomous experiments.

**Re-run all cells to refresh with latest data.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import subprocess

plt.style.use("dark_background")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["font.size"] = 12

In [ ]:
df = pd.read_csv("results.tsv", sep="\t")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["experiment"] = range(1, len(df) + 1)
kept = df[df["status"] == "keep"]
discarded = df[df["status"] == "discard"]
crashed = df[df["status"] == "crash"]
print(f"Total: {len(df)} | Kept: {len(kept)} | Discarded: {len(discarded)} | Crashed: {len(crashed)}")
print(f"Best composite: {df.composite.max():.2f} | Best tok/s: {df.avg_tps.max():.1f} | Best quality: {df.quality.max():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, title in zip(axes, ["composite", "avg_tps", "quality"], ["Composite Score", "Tokens/sec", "Quality"]):
    ax.scatter(discarded["experiment"], discarded[col], c="red", alpha=0.5, s=40, label="Discarded")
    ax.scatter(crashed["experiment"], crashed[col], c="gray", alpha=0.5, s=40, marker="x", label="Crashed")
    ax.scatter(kept["experiment"], kept[col], c="lime", s=80, zorder=5, label="Kept")
    if len(kept) > 1:
        ax.plot(kept["experiment"], kept[col], c="lime", alpha=0.5, linestyle="--")
    ax.set_xlabel("Experiment #")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("evolution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
cols = ["experiment", "composite", "avg_tps", "quality", "mem_eff", "vram_gb", "status", "description"]
df[cols]

In [ ]:
r = subprocess.run(["git", "log", "--oneline", "-20"], capture_output=True, text=True)
print("Git History (winning configs):")
print(r.stdout)

In [ ]:
if len(kept) > 0:
    best = kept.loc[kept.composite.idxmax()]
    print("CURRENT BEST CONFIG")
    print("=" * 40)
    for c in ["composite", "avg_tps", "quality", "mem_eff", "vram_gb", "commit", "description"]:
        print(f"  {c:12s}: {best[c]}")
else:
    print("No kept experiments yet. Run baseline first!")